# BTW End-to-End Bulk Transcriptomics Pipeline (Hybrid R/Python Architecture)
**Downstream Bulk Transcriptomics Workbench (`btw`) v0.2.0**

This comprehensive Jupyter notebook demonstrates downstream bulk RNA-seq analysis following the **SRS v2 specification ("Hybrid R/Python on Jupyter")**:
1. **Data I/O & Schema Validation (FR-1):** Ingest raw Count Matrix and Metadata with automated alignment and strict validation.
2. **Quality Control & Filtering (FR-2):** Assess library sizes, detection rates, and filter low-expression genes.
3. **Normalization (FR-2):** Median-of-Ratios size factor estimation with engine routing (`engine='r'` or `'python'`).
4. **Differential Expression Analysis (FR-3 & FR-10):** Wald test with hybrid engine dispatch (R Bioconductor `DESeq2` / `limma` or Python `PyDESeq2`), preserving native object handles.
5. **Publication Visualizations (FR-4):** Generate high-DPI figures (Volcano with adjustText, PCA with variance explained, and Clustered Heatmap).
6. **Functional & Pathway Enrichment (FR-5):** ORA (`clusterProfiler` / Enrichr / Fisher exact), GSEA Prerank, and Activity Inference.
7. **Advanced Downstream Analysis (FR-6, FR-7, FR-8):** ComBat batch correction (`sva::ComBat` or `inmoose`), WGCNA co-expression network (`WGCNA` or Python), and Cytoscape SIF export.
8. **Executive Reporting & Publication Deliverable Bundle (FR-9):** Generate self-contained HTML reports with engine provenance badges, Markdown summary, multi-sheet Excel workbook, and high-res vector figures.
9. **Acceptance Criteria Verification:** Verify unmasked access to reference objects (PyDESeq2 DeseqDataSet/DeseqStats, scikit-learn PCA, NetworkX Graph).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from sklearn.decomposition import PCA
import networkx as nx

# BTW core and R interop imports
import btw
from btw import set_seed, logger
from btw.r_interop import is_r_available, get_r_version, check_r_package
from btw.io import load_dataset, validate_bulk_data, export_table
from btw.qc_normalize import compute_sample_qc, compute_gene_qc, filter_low_expression_genes, normalize_deseq2
from btw.de_analysis import run_de, run_multiple_contrasts
from btw.viz import set_publication_style, plot_volcano, plot_pca, compute_pca, plot_heatmap, plot_ma, save_figure
from btw.enrichment import run_clusterprofiler, run_custom_ora, plot_enrichment_dotplot, plot_enrichment_barplot, prepare_ranked_gene_list
from btw.batch_correction import run_combat, compare_pca_batch, evaluate_batch_effect
from btw.network import detect_coexpression_modules, run_wgcna, module_to_networkx, export_cytoscape_sif, export_edge_list
from btw.reporting import export_publication_bundle, generate_html_report, generate_markdown_report

# Synchronize random seed across Python (NumPy/random) and R (set.seed)
set_seed(42)
set_publication_style(palette="nature", dpi=150)
print(f"Bulk Transcriptomics Workbench (BTW) v{btw.__version__}")
print(f"R Session Available: {is_r_available()} (Version: {get_r_version()})")

## 1. Synthetic Bulk RNA-seq Dataset Generation
Generate a synthetic Count Matrix (100 genes x 6 samples) with injected differential expression signals and simulated batch effects.

In [ ]:
genes = [f"GENE_{i:03d}" for i in range(1, 101)]
samples = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "treat_3"]

np.random.seed(42)
base_counts = np.random.negative_binomial(5, 0.01, size=(100, 6))
# Up-regulated genes in treated samples
base_counts[:15, 3:] = (base_counts[:15, 3:] * 4.5).astype(int)
# Down-regulated genes in treated samples
base_counts[15:30, 3:] = (base_counts[15:30, 3:] * 0.2).astype(int)

raw_counts = pd.DataFrame(base_counts, index=genes, columns=samples)
sample_metadata = pd.DataFrame({
    "sample_id": samples,
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch2", "batch1", "batch2", "batch1", "batch2"],
}).set_index("sample_id")

output_dir = Path("results/e2e_pipeline")
output_dir.mkdir(parents=True, exist_ok=True)
counts_path = output_dir / "raw_counts.csv"
meta_path = output_dir / "sample_metadata.csv"
export_table(raw_counts, counts_path, index=True)
export_table(sample_metadata, meta_path, index=True)
print(f"Synthetic assay saved to: {output_dir.resolve()}")

## 2. Data Ingestion & Schema Validation (FR-1)
Load raw counts and metadata through `load_dataset` with strict validation.

In [ ]:
dataset = load_dataset(counts_path, meta_path)
print(f"Loaded dataset: {dataset.n_genes} genes, {dataset.n_samples} samples")

val_report = dataset.validation_report or validate_bulk_data(dataset.counts, dataset.metadata)
print(val_report.summary())


## 3. Sample QC & Gene Filtering (FR-2)

In [ ]:
sample_qc = compute_sample_qc(dataset.counts)
print("Sample QC Metrics:")
print(sample_qc[["library_size", "detected_genes", "detection_rate_pct"]])

# Filter low-expression genes
filtered_counts, filter_summary = filter_low_expression_genes(dataset.counts, min_counts=5, min_samples=2)
print("\nFiltering Summary:")
print(filter_summary)

## 4. Count Normalization with Hybrid Engine Dispatch (FR-2 & FR-10)
Compute size factors via Median-of-Ratios. Supports reference R DESeq2 engine with automatic Python fallback.

In [ ]:
norm_result = normalize_deseq2(filtered_counts, engine="r", fallback_to_python=True)
print(f"Active Normalization Engine: {norm_result.engine.upper()}")
print("Estimated size factors:")
print(norm_result.size_factors)
print(f"Normalized count matrix shape: {norm_result.normalized_counts.shape}")

## 5. Differential Expression Analysis with Engine Routing (FR-3 & FR-10)
Execute Wald test differential expression analysis. Select between reference R `DESeq2` / `limma` or Python `PyDESeq2`.

In [ ]:
# Run DE with R reference engine and graceful Python fallback
de_result = run_de(
    counts=filtered_counts,
    metadata=dataset.metadata,
    contrast=("condition", "treated", "control"),
    engine="r",
    method="deseq2",
    fallback_to_python=True,
    alpha=0.05,
    lfc_threshold=1.0,
)

print(de_result.summary())

top_degs = de_result.get_degs().head(5)
print("\nTop 5 DEGs:")
print(top_degs[["baseMean", "log2FoldChange", "pvalue", "padj", "regulation"]])

## 6. Publication Visualizations (FR-4)
Generate Volcano Plot, PCA Plot, and Clustered Heatmap.

In [ ]:
fig_dir = output_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Volcano Plot
volcano_fig = plot_volcano(de_result, top_n_labels=10, title="Volcano Plot: Treated vs Control")
volcano_path = fig_dir / "volcano_plot.png"
save_figure(volcano_fig, volcano_path)

# 2. PCA Plot
pca_fig = plot_pca(norm_result, dataset.metadata, color_by="condition", shape_by="batch", title="PCA: Sample Segregation")
pca_path = fig_dir / "pca_plot.png"
save_figure(pca_fig, pca_path)

# 3. Clustered Heatmap
heatmap_fig = plot_heatmap(norm_result, dataset.metadata, de_result=de_result, top_n_degs=30, annotation_cols=["condition", "batch"], title="Top 30 DEGs Heatmap")
heatmap_path = fig_dir / "heatmap.png"
save_figure(heatmap_fig, heatmap_path)

print(f"Saved publication figures to: {fig_dir.resolve()}")

## 7. Pathway & Functional Enrichment Analysis (FR-5 & FR-10)
Analyze biological pathways via `run_clusterprofiler` (with fallback to Python custom ORA) and plot results.

In [ ]:
up_genes = de_result.get_up_genes()
down_genes = de_result.get_down_genes()

gene_sets = {
    "HALLMARK_INFLAMMATORY_RESPONSE": up_genes[:8] + ["GENE_099"],
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION": down_genes[:8] + ["GENE_098"],
    "HALLMARK_TNFA_SIGNALING": up_genes[4:12],
    "KEGG_CELL_CYCLE": [f"GENE_{i:03d}" for i in range(50, 65)],
}

enr_result = run_clusterprofiler(
    gene_list=de_result,
    direction="up",
    fallback_to_python=True,
    fallback_gene_sets=gene_sets,
    background=list(filtered_counts.index),
)
print(enr_result.summary())

dot_fig = plot_enrichment_dotplot(enr_result, top_n=5, title="Pathway Enrichment Dotplot")
dot_path = fig_dir / "enrichment_dotplot.png"
save_figure(dot_fig, dot_path)

## 8. Batch Effect Correction & Co-expression Networks (FR-7 & FR-8)
Perform batch correction with ComBat and co-expression analysis with WGCNA.

In [ ]:
# Batch correction via ComBat (R sva::ComBat or Python inmoose)
combat_counts = run_combat(
    data=norm_result.normalized_counts,
    batch="batch",
    metadata=dataset.metadata,
    biological_factor="condition",
    engine="r",
    fallback_to_python=True,
)
print(f"ComBat corrected shape: {combat_counts.shape}")

# Co-expression network analysis (R WGCNA or Python)
wgcna_res = run_wgcna(
    data=norm_result.normalized_counts.T,
    power=6,
    min_module_size=5,
    engine="r",
    fallback_to_python=True,
)
print(wgcna_res.summary())

# Export Cytoscape network
top_module = wgcna_res.modules[0] if wgcna_res.modules else 'module_1'
G = module_to_networkx(wgcna_res, module=top_module, threshold=0.01)
sif_path = export_cytoscape_sif(G, output_dir / f"wgcna_{top_module}.sif", interaction_type="coexpressed")
edge_path = export_edge_list(G, output_dir / f"wgcna_{top_module}_edges.tsv")
print(f"Exported Cytoscape network to {sif_path}")

## 9. Publication Deliverable Bundle (FR-9)
Consolidate all outputs into a unified publication bundle including HTML and Markdown reports with engine provenance badges.

In [ ]:
bundle_dir = output_dir / "publication_deliverable"
figure_dict = {
    "Volcano Plot": volcano_path,
    "PCA Plot": pca_path,
    "Clustered Heatmap": heatmap_path,
    "Enrichment Dotplot": dot_path,
}

bundle_path = export_publication_bundle(
    de_result=de_result,
    output_dir=bundle_dir,
    enrichment_results={"Hallmark_Pathways": enr_result},
    figure_paths=figure_dict,
    bundle_name="Bulk_RNA_End_to_End_Study_Report",
)

print(f"Publication bundle assembled successfully at: {bundle_path.resolve()}")
for f in sorted(bundle_dir.rglob("*")):
    if f.is_file():
        print(f"  - {f.relative_to(bundle_dir)} ({f.stat().st_size:,} bytes)")

## 10. Acceptance Criteria Verification
Confirm that native objects (PyDESeq2 DeseqDataSet/DeseqStats, scikit-learn PCA, NetworkX Graph) are directly accessible without restrictive wrapping.

In [ ]:
# 1. Native DE objects (PyDESeq2 or R handle)
if de_result.engine == "python":
    assert isinstance(de_result.dds, DeseqDataSet), "Underlying dds must be a DeseqDataSet"
    assert isinstance(de_result.stat_res, DeseqStats), "Underlying stat_res must be a DeseqStats"
    assert hasattr(de_result.dds, "varm"), "DeseqDataSet varm slot must be accessible"
    print("Acceptance 1 Passed: Direct access to PyDESeq2 DeseqDataSet and DeseqStats verified.")
else:
    assert de_result.dds is not None, "R native object handle must be accessible"
    print(f"Acceptance 1 Passed: Direct access to R native object ({type(de_result.dds)}) verified.")

# 2. Scikit-learn PCA Estimator
pca_model, pca_coords = compute_pca(norm_result.normalized_counts, dataset.metadata)
assert isinstance(pca_model, PCA), "Underlying pca_model must be a sklearn PCA object"
assert hasattr(pca_model, "explained_variance_ratio_"), "PCA explained variance must be directly accessible"
print("Acceptance 2 Passed: Direct access to Scikit-learn PCA estimator verified.")

# 3. NetworkX Co-expression Graph
assert isinstance(G, nx.Graph), "Network graph must be an instance of networkx.Graph"
print("Acceptance 3 Passed: Direct access to NetworkX graph object verified.")

print("\n🎉 ALL ACCEPTANCE CRITERIA SUCCESSFULLY VERIFIED!")